# 04 · SI biology

Walk-through of the Supplementary biology analyses of DisAtlas: cluster map and validation, rare–common transfer,
FDA-approved repurposing and evidence channels, event-time (prospective) evaluation, and compositionality.
Every cell reads shipped tables only (`data/raw`, `data/derived`, `data/zenodo`); the scripts under `figures/SI/`
regenerate these tables and the SI figures, and `tests/test_sibio_parity.py` checks them against the manuscript.

In [ ]:
from pathlib import Path
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DER, ZEN = ROOT / "data" / "derived", ROOT / "data" / "zenodo" / "si_bio"
tsv = lambda p, **k: pd.read_csv(p, sep="\t", **k)
pd.set_option("display.width", 160, "display.max_columns", 20)

## 1 · Cluster map and validation
K = 55 clusters on the 13,021-disease embedding: internal separation, K selection by bootstrap, and agreement between the 2-D layout and model distances.

In [ ]:
V = json.load(open(DER / "si_cluster" / "sifu_cluster_13021_values.json"))
pd.Series(V["internal"]).round(3)

In [ ]:
K = tsv(DER / "si_cluster" / "k_selection_boot.tsv.gz")
ks = K[K.kind == "boot"].groupby("K")[["silhouette", "ari_vs_full"]].agg(["mean", "std"]).round(3)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, col in zip(ax, ["silhouette", "ari_vs_full"]):
    a.errorbar(ks.index, ks[(col, "mean")], yerr=ks[(col, "std")], marker="o", ms=3); a.set_xlabel("K"); a.set_ylabel(col)
fig.tight_layout(); ks.head(12)

In [ ]:
WB = tsv(DER / "si_cluster" / "within_between_K55_13021.tsv.gz")
fig, ax = plt.subplots(figsize=(4.5, 3)); ax.hist(WB.silhouette, bins=60); ax.axvline(0, c="k", lw=.8)
ax.set_xlabel("per-disease silhouette"); ax.set_ylabel("diseases")
z = np.load(ZEN / "layout_pairs_13021.npz")
print("diseases", len(WB), "| negative silhouette share", round((WB.silhouette < 0).mean(), 3),
      "| layout vs model distance Spearman (sampled pairs)", round(pd.Series(z["dmap"]).corr(pd.Series(z["dmod"]), method="spearman"), 3))

## 2 · Rare–common transfer
How often a rare disease has a common disease among its nearest neighbours, against annotation-, degree- and uniform-preserving nulls, and benchmark AUROC by rare/common stratum.

In [ ]:
Cv = tsv(DER / "si_rare_common" / "coverage_by_k_summary.tsv")
fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for a, cls in zip(ax, ["rare", "common"]):
    for m, g in Cv[Cv.anchor_class == cls].groupby("model"):
        g = g[g.k <= 50]; a.plot(g.k, g["mean"], label=m); a.fill_between(g.k, g.lo, g.hi, alpha=.2)
    a.set_title(f"{cls} anchors"); a.set_xlabel("k nearest neighbours"); a.set_ylabel("share with other-class neighbour")
ax[0].legend(fontsize=7); fig.tight_layout()
json.load(open(DER / "si_rare_common" / "rc_coverage_meta.json"))

In [ ]:
S = tsv(DER / "si_rare_common" / "ccplus_strata.tsv")
S[["benchmark", "scope", "stratum", "n_pairs", "n_diseases", "auroc", "ci_lo", "ci_hi", "delta_vs_stack"]].round(3)

## 3 · FDA-approved repurposing and evidence channels
Development split only. Disease pairs whose approval evidence comes from licensed sources carry blinded disease tokens (`T_…`); scores and pair classes are unchanged.

In [ ]:
B = tsv(DER / "si_fda" / "benchmark_pairs_split.tsv")
print("pairs", len(B), "| diseases", len(set(B.mondo_a) | set(B.mondo_b)),
      "| blinded disease tokens", sum(str(x).startswith("T_") for x in set(B.mondo_a) | set(B.mondo_b)))
B.groupby([c for c in B.columns if c in ("split", "split_group")][0]).size()

In [ ]:
R = tsv(DER / "si_fda" / "broad_only_fda_dev_results.tsv")
t = R[(R.metric == "recovered") & (R.pair_set == "own")][["arm_label", "n_scored", "value", "ci_lo", "ci_hi"]].round(3)
fig, ax = plt.subplots(figsize=(5, 2.6)); ax.barh(t.arm_label, t.value, xerr=[t.value - t.ci_lo, t.ci_hi - t.value]); ax.set_xlabel("recovery (percentile ≥ 75)")
fig.tight_layout(); t

In [ ]:
P = tsv(DER / "si_fda" / "broad_only_fda_dev_pair_scores_ens5.tsv")
P.groupby("pair_class").size().sort_values(ascending=False)

## 4 · Event-time (prospective) evaluation
For each drug and decision year, conditional logistic regression (one stratum per drug and year, popularity spline, Efron ties, drug-clustered 95% CIs) of which not-yet-trialed held-out disease the drug was trialed in next, per 1 SD of each method's score (Figure 7b–c). M0 is DisAtlas without popularity adjustment, PERM_M2 the shuffled-score control.

In [ ]:
OR = tsv(DER / "fig7_prospective" / "or_clogit.tsv", dtype={"set": str})
p = OR[OR.set == "pooled"][["model", "term", "OR", "lo95", "hi95", "n_cases", "n_drugs"]].round(3)
p.sort_values("OR", ascending=False)

In [ ]:
y = OR[(OR.model == "M2") & OR.set.str.isdigit()].assign(yr=lambda d: d.set.astype(int)).query("yr >= 2000")
fig, ax = plt.subplots(figsize=(5, 3)); ax.errorbar(y.yr, y.OR, yerr=[y.OR - y.lo95, y.hi95 - y.OR], fmt="o")
ax.axhline(1, c="k", lw=.8); ax.set_xlabel("decision year"); ax.set_ylabel("odds ratio per 1 SD (DisAtlas, adjusted)"); fig.tight_layout()

## 5 · Compositionality
Whether the barycentre of a disease's parent diseases lands near the disease itself, for worked examples and systematically over MONDO multi-parent diseases.

In [ ]:
E = tsv(DER / "si_compositionality" / "examples.tsv")
display(E[["example", "n_components", "rank", "out_of", "null_median_rank", "p"]])
tsv(DER / "si_compositionality" / "systematic_summary.tsv").round(3)

In [ ]:
json.load(open(DER / "si_compositionality" / "compositionality_summary.json"))